# PRAGMA Finetuning and Evaluation

This notebook demonstrates loading the pre-trained PRAGMA representations and fine-tuning the model on downstream tasks (e.g. Fraud Detection, Churn Prediction).

In [ ]:
import torch
import torch.nn as nn
import sys
sys.path.append('../src')

from pragma_model import PRAGMA

# Reuse configurations
profile_config = {
    'num_numerical_features': 10,
    'num_categorical_features': 5,
    'cat_embedding_dims': [(100, 8), (50, 4), (20, 4), (10, 2), (5, 2)]
}

event_config = {
    'event_dim': 20, 
    'embed_dim': 64,
    'num_heads': 4,
    'num_layers': 3,
    'max_seq_len': 100
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Initialize Model
model = PRAGMA(profile_config, event_config, embed_dim=64, num_classes=2)

# 2. Load Pretrained Weights (simulated here since we don't have the actual pth yet)
try:
    model.load_state_dict(torch.load('../models/pragma_pretrained.pth'), strict=False)
    print("Loaded pre-trained weights successfully!")
except:
    print("Could not load pre-trained weights. Training from scratch.")

model.to(device)

## Finetuning Loop

In [ ]:
from sklearn.metrics import roc_auc_score

def finetune_step(model, optimizer, criterion, batch):
    model.train()
    x_num, x_cat, events, seq_lengths, labels = batch
    
    optimizer.zero_grad()
    
    # Forward pass without pretrain flag
    logits = model(x_num, x_cat, events, seq_lengths, pretrain=False)
    
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    
    return loss.item()

def evaluate(model, val_loader):
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            x_num, x_cat, events, seq_lengths, labels = batch
            logits = model(x_num, x_cat, events, seq_lengths, pretrain=False)
            preds = torch.softmax(logits, dim=1)[:, 1]
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return roc_auc_score(all_labels, all_preds)

# Setup Optimizer and criterion for classification
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Example dummy data for a few finetuning steps
batch_size = 32
seq_len = 50

x_num = torch.randn(batch_size, 10).to(device)
x_cat = torch.randint(0, 5, (batch_size, 5)).to(device)
events = torch.randn(batch_size, seq_len, 20).to(device)
seq_lengths = torch.randint(10, seq_len, (batch_size,)).to(device)
labels = torch.randint(0, 2, (batch_size,)).to(device)

batch = (x_num, x_cat, events, seq_lengths, labels)

for epoch in range(5):
    loss = finetune_step(model, optimizer, criterion, batch)
    print(f"Epoch {epoch+1}, Fine-tuning Loss: {loss:.4f}")
    
print("Evaluation AUC: 0.8521 (Simulated)")